# Data preparation

Run the cells in order, top to bottom. Each one uses what the previous one wrote,
so do not skip or reorder. Stop at the first error.

## 0. Setup

Points the notebook at the data folder and the right Python.

In [ ]:
import os, sys, subprocess

DATA_DIR = r"E:\mimic_data"
PYTHON = sys.executable
os.chdir(DATA_DIR)

def run(script):
    print(">>>", script, "\n")
    p = subprocess.run([PYTHON, script], capture_output=True, text=True)
    print(p.stdout)
    if p.stderr:
        print("--- stderr ---")
        print(p.stderr)
    print("exit code:", p.returncode)
    return p.returncode

print("folder :", os.getcwd())
print("python :", PYTHON)

## 1. Which images are still missing

Checks every file on disk for the JPEG end-of-image marker, deletes anything
truncated, and writes missing.txt. Files still listed after a finished download
are ones that do not exist in MIMIC-CXR-JPG v2.0.0 at all.

In [ ]:
run("make_missing.py")

## 2. Build the usable subset

Keeps only the rows whose image is on disk and complete, and writes
available_labels.csv with the train / validate / test split.

In [ ]:
run("build_available_subset.py")

## 3. Extract the reports

Reads the report archive, splits out FINDINGS and IMPRESSION, and writes
study_reports.csv and image_report_pairs.csv.

In [ ]:
run("prepare_reports.py")

## 4. Delete the old backup

clean_reports.py always reads image_report_pairs_raw.csv when it exists, so an
old copy would silently freeze the dataset at its previous size. This cell
removes it so the backup is written fresh from the new data.

In [ ]:
import os
p = "image_report_pairs_raw.csv"
if os.path.exists(p):
    os.remove(p)
    print("deleted", p)
else:
    print("not present, nothing to delete")

## 5. Clean the reports

Removes sentences that compare against a previous study, since they cannot be
inferred from a single image.

In [ ]:
run("clean_reports.py")

## 6. Summary

Final counts, to compare against the previous run.

In [ ]:
import pandas as pd

lab = pd.read_csv("available_labels.csv")
pairs = pd.read_csv("image_report_pairs.csv")

w = pairs["report"].fillna("").astype(str).str.split().str.len()
print("pairs        :", len(pairs))
print("mean words   : %.1f" % w.mean())

merged = lab.merge(pairs[["dicom_id"]], on="dicom_id", how="inner")
print("\nimages that have a report")
print(merged["split"].value_counts().to_string())
print("  total :", len(merged))

## Next

Counts look right, so retrain the classifier with the new cxr_data.py
(320 px, padded instead of cropped, no horizontal flip), then regenerate the
prompts and retrain the generator.